# MCP Tools

**Module:** 12-mcp

**Notebook:** `04-mcp-tools.ipynb`

This expanded lesson goes beyond definitions: each topic includes *why it matters*, *how it works*, intuition, pitfalls, and when to use it—plus runnable Python demos, comparison aids, and exercises.


## Learning Objectives

By the end of this notebook, you will be able to:

- Explain and apply **What are MCP Tools?** with clear contracts and failure modes
- Explain and apply **Listing Tools** with clear contracts and failure modes
- Explain and apply **Calling Tools** with clear contracts and failure modes
- Explain and apply **Schemas & Validation** with clear contracts and failure modes
- Explain and apply **Side Effects** with clear contracts and failure modes
- Explain and apply **Error Model** with clear contracts and failure modes
- Explain and apply **Host Integration Pattern** with clear contracts and failure modes
- Evaluate tradeoffs (quality, cost, latency, safety) for designs in this lesson
- Implement small Python prototypes that make the ideas testable


## How to Use This Notebook

1. Read the topic sections fully—do not jump only to code.
2. Run each demo; then change inputs to break them and fix them.
3. API examples use placeholders like `YOUR_API_KEY` or `os.environ.get(...)`.
4. Keep secrets out of git; treat prompts/tool schemas as versioned code.
5. Complete the **Try It Yourself** exercises before moving on.


### Pipeline walkthrough — MCP Tools

```mermaid
flowchart LR
  A[Problem / user goal] --> B[Contract: IO + constraints]
  B --> C[Implement core path]
  C --> D[Validate / guardrails]
  D --> E[Eval fixtures]
  E --> F[Observe in production]
  F -->|regressions| B
```

```text
goal -> contract -> implement -> validate -> evaluate -> monitor -> revise
```


## Curriculum Map

This notebook's spine (preserve/cover all of these):

1. **What are MCP Tools?**
2. **Listing Tools**
3. **Calling Tools**
4. **Schemas & Validation**
5. **Side Effects**
6. **Error Model**
7. **Host Integration Pattern**

Read top-to-bottom once, then revisit weak spots with the exercises.


## What are MCP Tools?

### Definition
**What are MCP Tools?** is a core building block in 04-mcp-tools within Model Context Protocol (MCP). Treat it as a plugin protocol for context and actions (think LSP for model tools): something you can name, version, test, and operate.

### Why it matters
In Model Context Protocol (MCP), weak designs around What are MCP Tools? typically surface as overexposed tools, confused trust boundaries, and brittle transports. Investing here improves reliability, debuggability, and the ability to change models later.

### How it works
For What are MCP Tools?: (1) write an explicit input/output contract, (2) implement the minimal happy path, (3) validate and add guardrails, (4) cover golden + adversarial fixtures, (5) wire observability. Your durable artifacts should look like servers, resources, tools, prompts, and transports.

### Intuition
Explain What are MCP Tools? as a plugin protocol for context and actions (think LSP for model tools). If a new engineer cannot tell what is trusted input, what is allowed action, and what 'done' means, the design is still fuzzy.

### Pitfalls
- Treating What are MCP Tools? as a one-time playground experiment instead of a versioned artifact
- No success criteria or eval set for What are MCP Tools?
- Ignoring cost/latency tradeoffs while chasing marginal quality
- Missing adversarial cases typical of Model Context Protocol (MCP): overexposed tools, confused trust boundaries, and brittle transports

### When to use
Use What are MCP Tools? when your product path depends on this concern in Model Context Protocol (MCP). Prefer the simplest design that meets quality, latency, and safety budgets—and prove it with fixtures.

### Quick reference

| Lens | Question |
|------|----------|
| Product | What user outcome does What are MCP Tools? improve? |
| Engineering | What is the interface / data contract? |
| Safety | What can go wrong if it fails open? |
| Ops | How will we notice regressions? |


In [ ]:
# Demo: make "What are MCP Tools?" concrete as a checkable contract
from dataclasses import dataclass, field, asdict
import json

@dataclass
class ConceptContract:
    name: str = "What are MCP Tools?"
    notebook: str = "04-mcp-tools"
    must_have: list = field(default_factory=lambda: [
        "clear inputs/outputs",
        "failure behavior defined",
        "eval fixtures exist",
    ])
    risks: list = field(default_factory=lambda: [
        "silent quality drift",
        "unbounded cost/latency",
    ])

    def health(self) -> dict:
        return {
            "concept": self.name,
            "checks": len(self.must_have),
            "risks": len(self.risks),
            "ready_for_design_review": len(self.must_have) >= 3,
        }

contract_0 = ConceptContract()
print(json.dumps({"contract": asdict(contract_0), "health": contract_0.health()}, indent=2))


In [ ]:
import json

import ast, operator
_OPS = {ast.Add: operator.add, ast.Sub: operator.sub, ast.Mult: operator.mul, ast.Div: operator.truediv}

def _safe_calc(expr: str):
    node = ast.parse(expr, mode="eval")
    def ev(n):
        if isinstance(n, ast.Expression): return ev(n.body)
        if isinstance(n, ast.Constant) and isinstance(n.value, (int, float)): return n.value
        if isinstance(n, ast.BinOp) and type(n.op) in _OPS: return _OPS[type(n.op)](ev(n.left), ev(n.right))
        raise ValueError("unsupported")
    return ev(node)

TOOLS = {
    "search_docs": lambda q: [{"id": "d1", "text": f"Snippet for {q}"}],
    "safe_calc": lambda expr: {"result": _safe_calc(expr)},
}

def route(name: str, args_json: str) -> dict:
    if name not in TOOLS:
        return {"ok": False, "error": "unknown_tool"}
    try:
        args = json.loads(args_json)
        return {"ok": True, "observation": TOOLS[name](**args)}
    except Exception as e:
        return {"ok": False, "error": type(e).__name__}

print(route("search_docs", '{"q":"SSO"}'))
print(route("safe_calc", '{"expr":"21*2"}'))


In [ ]:
# ReAct-style trace (pedagogical)
trace = [
    ("Thought", "Need docs on SSO redirects"),
    ("Action", "search_docs"),
    ("Args", {"q": "SSO redirect allowlist"}),
    ("Observation", route("search_docs", '{"q":"SSO redirect allowlist"}')),
    ("Final", "Redirect URLs must match the allowlist."),
]
for k, v in trace:
    print(f"{k}: {v}")


## Listing Tools

### Definition
**Listing Tools** is a core building block in 04-mcp-tools within Model Context Protocol (MCP). Treat it as a plugin protocol for context and actions (think LSP for model tools): something you can name, version, test, and operate.

### Why it matters
In Model Context Protocol (MCP), weak designs around Listing Tools typically surface as overexposed tools, confused trust boundaries, and brittle transports. Investing here improves reliability, debuggability, and the ability to change models later.

### How it works
For Listing Tools: (1) write an explicit input/output contract, (2) implement the minimal happy path, (3) validate and add guardrails, (4) cover golden + adversarial fixtures, (5) wire observability. Your durable artifacts should look like servers, resources, tools, prompts, and transports.

### Intuition
Explain Listing Tools as a plugin protocol for context and actions (think LSP for model tools). If a new engineer cannot tell what is trusted input, what is allowed action, and what 'done' means, the design is still fuzzy.

### Pitfalls
- Treating Listing Tools as a one-time playground experiment instead of a versioned artifact
- No success criteria or eval set for Listing Tools
- Ignoring cost/latency tradeoffs while chasing marginal quality
- Missing adversarial cases typical of Model Context Protocol (MCP): overexposed tools, confused trust boundaries, and brittle transports

### When to use
Use Listing Tools when your product path depends on this concern in Model Context Protocol (MCP). Prefer the simplest design that meets quality, latency, and safety budgets—and prove it with fixtures.


In [ ]:
# Demo: make "Listing Tools" concrete as a checkable contract
from dataclasses import dataclass, field, asdict
import json

@dataclass
class ConceptContract:
    name: str = "Listing Tools"
    notebook: str = "04-mcp-tools"
    must_have: list = field(default_factory=lambda: [
        "clear inputs/outputs",
        "failure behavior defined",
        "eval fixtures exist",
    ])
    risks: list = field(default_factory=lambda: [
        "silent quality drift",
        "unbounded cost/latency",
    ])

    def health(self) -> dict:
        return {
            "concept": self.name,
            "checks": len(self.must_have),
            "risks": len(self.risks),
            "ready_for_design_review": len(self.must_have) >= 3,
        }

contract_1 = ConceptContract()
print(json.dumps({"contract": asdict(contract_1), "health": contract_1.health()}, indent=2))


In [ ]:
import json

import ast, operator
_OPS = {ast.Add: operator.add, ast.Sub: operator.sub, ast.Mult: operator.mul, ast.Div: operator.truediv}

def _safe_calc(expr: str):
    node = ast.parse(expr, mode="eval")
    def ev(n):
        if isinstance(n, ast.Expression): return ev(n.body)
        if isinstance(n, ast.Constant) and isinstance(n.value, (int, float)): return n.value
        if isinstance(n, ast.BinOp) and type(n.op) in _OPS: return _OPS[type(n.op)](ev(n.left), ev(n.right))
        raise ValueError("unsupported")
    return ev(node)

TOOLS = {
    "search_docs": lambda q: [{"id": "d1", "text": f"Snippet for {q}"}],
    "safe_calc": lambda expr: {"result": _safe_calc(expr)},
}

def route(name: str, args_json: str) -> dict:
    if name not in TOOLS:
        return {"ok": False, "error": "unknown_tool"}
    try:
        args = json.loads(args_json)
        return {"ok": True, "observation": TOOLS[name](**args)}
    except Exception as e:
        return {"ok": False, "error": type(e).__name__}

print(route("search_docs", '{"q":"SSO"}'))
print(route("safe_calc", '{"expr":"21*2"}'))


In [ ]:
# ReAct-style trace (pedagogical)
trace = [
    ("Thought", "Need docs on SSO redirects"),
    ("Action", "search_docs"),
    ("Args", {"q": "SSO redirect allowlist"}),
    ("Observation", route("search_docs", '{"q":"SSO redirect allowlist"}')),
    ("Final", "Redirect URLs must match the allowlist."),
]
for k, v in trace:
    print(f"{k}: {v}")


### Worked scenario — Listing Tools

**Situation:** A team wants to productionize a feature involving **Listing Tools**.

**Walkthrough:**
1. Write a one-sentence success metric.
2. Define inputs, outputs, and hard constraints.
3. Implement the smallest demo that can fail loudly.
4. Add one adversarial fixture (empty, hostile, or oversized input).
5. Decide ship/no-ship using the metric—not eloquence.


## Calling Tools

### Definition
**Calling Tools** is a core building block in 04-mcp-tools within Model Context Protocol (MCP). Treat it as a plugin protocol for context and actions (think LSP for model tools): something you can name, version, test, and operate.

### Why it matters
In Model Context Protocol (MCP), weak designs around Calling Tools typically surface as overexposed tools, confused trust boundaries, and brittle transports. Investing here improves reliability, debuggability, and the ability to change models later.

### How it works
For Calling Tools: (1) write an explicit input/output contract, (2) implement the minimal happy path, (3) validate and add guardrails, (4) cover golden + adversarial fixtures, (5) wire observability. Your durable artifacts should look like servers, resources, tools, prompts, and transports.

### Intuition
Explain Calling Tools as a plugin protocol for context and actions (think LSP for model tools). If a new engineer cannot tell what is trusted input, what is allowed action, and what 'done' means, the design is still fuzzy.

### Pitfalls
- Treating Calling Tools as a one-time playground experiment instead of a versioned artifact
- No success criteria or eval set for Calling Tools
- Ignoring cost/latency tradeoffs while chasing marginal quality
- Missing adversarial cases typical of Model Context Protocol (MCP): overexposed tools, confused trust boundaries, and brittle transports

### When to use
Use Calling Tools when your product path depends on this concern in Model Context Protocol (MCP). Prefer the simplest design that meets quality, latency, and safety budgets—and prove it with fixtures.


In [ ]:
# Demo: make "Calling Tools" concrete as a checkable contract
from dataclasses import dataclass, field, asdict
import json

@dataclass
class ConceptContract:
    name: str = "Calling Tools"
    notebook: str = "04-mcp-tools"
    must_have: list = field(default_factory=lambda: [
        "clear inputs/outputs",
        "failure behavior defined",
        "eval fixtures exist",
    ])
    risks: list = field(default_factory=lambda: [
        "silent quality drift",
        "unbounded cost/latency",
    ])

    def health(self) -> dict:
        return {
            "concept": self.name,
            "checks": len(self.must_have),
            "risks": len(self.risks),
            "ready_for_design_review": len(self.must_have) >= 3,
        }

contract_2 = ConceptContract()
print(json.dumps({"contract": asdict(contract_2), "health": contract_2.health()}, indent=2))


In [ ]:
import json

import ast, operator
_OPS = {ast.Add: operator.add, ast.Sub: operator.sub, ast.Mult: operator.mul, ast.Div: operator.truediv}

def _safe_calc(expr: str):
    node = ast.parse(expr, mode="eval")
    def ev(n):
        if isinstance(n, ast.Expression): return ev(n.body)
        if isinstance(n, ast.Constant) and isinstance(n.value, (int, float)): return n.value
        if isinstance(n, ast.BinOp) and type(n.op) in _OPS: return _OPS[type(n.op)](ev(n.left), ev(n.right))
        raise ValueError("unsupported")
    return ev(node)

TOOLS = {
    "search_docs": lambda q: [{"id": "d1", "text": f"Snippet for {q}"}],
    "safe_calc": lambda expr: {"result": _safe_calc(expr)},
}

def route(name: str, args_json: str) -> dict:
    if name not in TOOLS:
        return {"ok": False, "error": "unknown_tool"}
    try:
        args = json.loads(args_json)
        return {"ok": True, "observation": TOOLS[name](**args)}
    except Exception as e:
        return {"ok": False, "error": type(e).__name__}

print(route("search_docs", '{"q":"SSO"}'))
print(route("safe_calc", '{"expr":"21*2"}'))


In [ ]:
# ReAct-style trace (pedagogical)
trace = [
    ("Thought", "Need docs on SSO redirects"),
    ("Action", "search_docs"),
    ("Args", {"q": "SSO redirect allowlist"}),
    ("Observation", route("search_docs", '{"q":"SSO redirect allowlist"}')),
    ("Final", "Redirect URLs must match the allowlist."),
]
for k, v in trace:
    print(f"{k}: {v}")


## Schemas & Validation

### Definition
**Schemas & Validation** is a core building block in 04-mcp-tools within Model Context Protocol (MCP). Treat it as a plugin protocol for context and actions (think LSP for model tools): something you can name, version, test, and operate.

### Why it matters
In Model Context Protocol (MCP), weak designs around Schemas & Validation typically surface as overexposed tools, confused trust boundaries, and brittle transports. Investing here improves reliability, debuggability, and the ability to change models later.

### How it works
For Schemas & Validation: (1) write an explicit input/output contract, (2) implement the minimal happy path, (3) validate and add guardrails, (4) cover golden + adversarial fixtures, (5) wire observability. Your durable artifacts should look like servers, resources, tools, prompts, and transports.

### Intuition
Explain Schemas & Validation as a plugin protocol for context and actions (think LSP for model tools). If a new engineer cannot tell what is trusted input, what is allowed action, and what 'done' means, the design is still fuzzy.

### Pitfalls
- Treating Schemas & Validation as a one-time playground experiment instead of a versioned artifact
- No success criteria or eval set for Schemas & Validation
- Ignoring cost/latency tradeoffs while chasing marginal quality
- Missing adversarial cases typical of Model Context Protocol (MCP): overexposed tools, confused trust boundaries, and brittle transports

### When to use
Use Schemas & Validation when your product path depends on this concern in Model Context Protocol (MCP). Prefer the simplest design that meets quality, latency, and safety budgets—and prove it with fixtures.


In [ ]:
# Demo: make "Schemas & Validation" concrete as a checkable contract
from dataclasses import dataclass, field, asdict
import json

@dataclass
class ConceptContract:
    name: str = "Schemas & Validation"
    notebook: str = "04-mcp-tools"
    must_have: list = field(default_factory=lambda: [
        "clear inputs/outputs",
        "failure behavior defined",
        "eval fixtures exist",
    ])
    risks: list = field(default_factory=lambda: [
        "silent quality drift",
        "unbounded cost/latency",
    ])

    def health(self) -> dict:
        return {
            "concept": self.name,
            "checks": len(self.must_have),
            "risks": len(self.risks),
            "ready_for_design_review": len(self.must_have) >= 3,
        }

contract_3 = ConceptContract()
print(json.dumps({"contract": asdict(contract_3), "health": contract_3.health()}, indent=2))


In [ ]:
import json, re

def extract_json(text: str):
    text = text.strip()
    m = re.search(r"```(?:json)?\s*([\s\S]*?)```", text)
    if m:
        text = m.group(1).strip()
    text = text.replace(",}", "}").replace(",]", "]")
    return json.loads(text)

schema_required = {"category", "priority"}
samples = [
    '{"category":"auth","priority":"P1"}',
    '```json\n{"category":"billing","priority":"P2",}\n```',
]
for s in samples:
    obj = extract_json(s)
    assert schema_required <= set(obj)
    print("ok:", obj)


In [ ]:
# Realistic API response_format shape (placeholder key)
import os
request = {
    "model": "gpt-4.1-mini",
    "messages": [
        {"role": "system", "content": "Return JSON only."},
        {"role": "user", "content": "Classify: SSO login loop"},
    ],
    "response_format": {"type": "json_object"},
}
headers = {"Authorization": f"Bearer {os.environ.get('OPENAI_API_KEY', 'YOUR_API_KEY')}"}
response = {
    "choices": [{"message": {"content": '{"category":"auth","priority":"P1"}'}}],
    "usage": {"prompt_tokens": 90, "completion_tokens": 12},
}
print(headers["Authorization"][:22] + "...", json.loads(response["choices"][0]["message"]["content"]))


### Worked scenario — Schemas & Validation

**Situation:** A team wants to productionize a feature involving **Schemas & Validation**.

**Walkthrough:**
1. Write a one-sentence success metric.
2. Define inputs, outputs, and hard constraints.
3. Implement the smallest demo that can fail loudly.
4. Add one adversarial fixture (empty, hostile, or oversized input).
5. Decide ship/no-ship using the metric—not eloquence.


## Side Effects

### Definition
**Side Effects** is a core building block in 04-mcp-tools within Model Context Protocol (MCP). Treat it as a plugin protocol for context and actions (think LSP for model tools): something you can name, version, test, and operate.

### Why it matters
In Model Context Protocol (MCP), weak designs around Side Effects typically surface as overexposed tools, confused trust boundaries, and brittle transports. Investing here improves reliability, debuggability, and the ability to change models later.

### How it works
For Side Effects: (1) write an explicit input/output contract, (2) implement the minimal happy path, (3) validate and add guardrails, (4) cover golden + adversarial fixtures, (5) wire observability. Your durable artifacts should look like servers, resources, tools, prompts, and transports.

### Intuition
Explain Side Effects as a plugin protocol for context and actions (think LSP for model tools). If a new engineer cannot tell what is trusted input, what is allowed action, and what 'done' means, the design is still fuzzy.

### Pitfalls
- Treating Side Effects as a one-time playground experiment instead of a versioned artifact
- No success criteria or eval set for Side Effects
- Ignoring cost/latency tradeoffs while chasing marginal quality
- Missing adversarial cases typical of Model Context Protocol (MCP): overexposed tools, confused trust boundaries, and brittle transports

### When to use
Use Side Effects when your product path depends on this concern in Model Context Protocol (MCP). Prefer the simplest design that meets quality, latency, and safety budgets—and prove it with fixtures.


In [ ]:
# Demo: make "Side Effects" concrete as a checkable contract
from dataclasses import dataclass, field, asdict
import json

@dataclass
class ConceptContract:
    name: str = "Side Effects"
    notebook: str = "04-mcp-tools"
    must_have: list = field(default_factory=lambda: [
        "clear inputs/outputs",
        "failure behavior defined",
        "eval fixtures exist",
    ])
    risks: list = field(default_factory=lambda: [
        "silent quality drift",
        "unbounded cost/latency",
    ])

    def health(self) -> dict:
        return {
            "concept": self.name,
            "checks": len(self.must_have),
            "risks": len(self.risks),
            "ready_for_design_review": len(self.must_have) >= 3,
        }

contract_4 = ConceptContract()
print(json.dumps({"contract": asdict(contract_4), "health": contract_4.health()}, indent=2))


In [ ]:
# Demo: before/after quality rubric for "Side Effects"
def score_artifact(artifact: dict, rubric: list[str]) -> dict:
    missing = [r for r in rubric if not artifact.get(r)]
    return {"score": round(1 - len(missing)/max(1,len(rubric)), 2), "missing": missing}

rubric = ["definition", "example", "failure_mode", "metric"]
weak = {"definition": "Side Effects"}
strong = {"definition": "Side Effects", "example": "worked example", "failure_mode": "empty input", "metric": "exact_match"}
print("weak", score_artifact(weak, rubric))
print("strong", score_artifact(strong, rubric))


In [ ]:
# Demo: operational checklist runner for "Side Effects"
checks = {
    "has_owner": True,
    "has_eval_set": True,
    "has_token_budget": False,
    "has_alert": False,
}
failed = [k for k, ok in checks.items() if not ok]
print({"topic": "Side Effects", "passed": len(checks)-len(failed), "failed": failed})


## Error Model

### Definition
**Error Model** is a core building block in 04-mcp-tools within Model Context Protocol (MCP). Treat it as a plugin protocol for context and actions (think LSP for model tools): something you can name, version, test, and operate.

### Why it matters
In Model Context Protocol (MCP), weak designs around Error Model typically surface as overexposed tools, confused trust boundaries, and brittle transports. Investing here improves reliability, debuggability, and the ability to change models later.

### How it works
For Error Model: (1) write an explicit input/output contract, (2) implement the minimal happy path, (3) validate and add guardrails, (4) cover golden + adversarial fixtures, (5) wire observability. Your durable artifacts should look like servers, resources, tools, prompts, and transports.

### Intuition
Explain Error Model as a plugin protocol for context and actions (think LSP for model tools). If a new engineer cannot tell what is trusted input, what is allowed action, and what 'done' means, the design is still fuzzy.

### Pitfalls
- Treating Error Model as a one-time playground experiment instead of a versioned artifact
- No success criteria or eval set for Error Model
- Ignoring cost/latency tradeoffs while chasing marginal quality
- Missing adversarial cases typical of Model Context Protocol (MCP): overexposed tools, confused trust boundaries, and brittle transports

### When to use
Use Error Model when your product path depends on this concern in Model Context Protocol (MCP). Prefer the simplest design that meets quality, latency, and safety budgets—and prove it with fixtures.


In [ ]:
# Demo: make "Error Model" concrete as a checkable contract
from dataclasses import dataclass, field, asdict
import json

@dataclass
class ConceptContract:
    name: str = "Error Model"
    notebook: str = "04-mcp-tools"
    must_have: list = field(default_factory=lambda: [
        "clear inputs/outputs",
        "failure behavior defined",
        "eval fixtures exist",
    ])
    risks: list = field(default_factory=lambda: [
        "silent quality drift",
        "unbounded cost/latency",
    ])

    def health(self) -> dict:
        return {
            "concept": self.name,
            "checks": len(self.must_have),
            "risks": len(self.risks),
            "ready_for_design_review": len(self.must_have) >= 3,
        }

contract_5 = ConceptContract()
print(json.dumps({"contract": asdict(contract_5), "health": contract_5.health()}, indent=2))


In [ ]:
# Demo: before/after quality rubric for "Error Model"
def score_artifact(artifact: dict, rubric: list[str]) -> dict:
    missing = [r for r in rubric if not artifact.get(r)]
    return {"score": round(1 - len(missing)/max(1,len(rubric)), 2), "missing": missing}

rubric = ["definition", "example", "failure_mode", "metric"]
weak = {"definition": "Error Model"}
strong = {"definition": "Error Model", "example": "worked example", "failure_mode": "empty input", "metric": "exact_match"}
print("weak", score_artifact(weak, rubric))
print("strong", score_artifact(strong, rubric))


In [ ]:
# Demo: operational checklist runner for "Error Model"
checks = {
    "has_owner": True,
    "has_eval_set": True,
    "has_token_budget": False,
    "has_alert": False,
}
failed = [k for k, ok in checks.items() if not ok]
print({"topic": "Error Model", "passed": len(checks)-len(failed), "failed": failed})


### Worked scenario — Error Model

**Situation:** A team wants to productionize a feature involving **Error Model**.

**Walkthrough:**
1. Write a one-sentence success metric.
2. Define inputs, outputs, and hard constraints.
3. Implement the smallest demo that can fail loudly.
4. Add one adversarial fixture (empty, hostile, or oversized input).
5. Decide ship/no-ship using the metric—not eloquence.


## Host Integration Pattern

### Definition
**Host Integration Pattern** is a core building block in 04-mcp-tools within Model Context Protocol (MCP). Treat it as a plugin protocol for context and actions (think LSP for model tools): something you can name, version, test, and operate.

### Why it matters
In Model Context Protocol (MCP), weak designs around Host Integration Pattern typically surface as overexposed tools, confused trust boundaries, and brittle transports. Investing here improves reliability, debuggability, and the ability to change models later.

### How it works
For Host Integration Pattern: (1) write an explicit input/output contract, (2) implement the minimal happy path, (3) validate and add guardrails, (4) cover golden + adversarial fixtures, (5) wire observability. Your durable artifacts should look like servers, resources, tools, prompts, and transports.

### Intuition
Explain Host Integration Pattern as a plugin protocol for context and actions (think LSP for model tools). If a new engineer cannot tell what is trusted input, what is allowed action, and what 'done' means, the design is still fuzzy.

### Pitfalls
- Treating Host Integration Pattern as a one-time playground experiment instead of a versioned artifact
- No success criteria or eval set for Host Integration Pattern
- Ignoring cost/latency tradeoffs while chasing marginal quality
- Missing adversarial cases typical of Model Context Protocol (MCP): overexposed tools, confused trust boundaries, and brittle transports

### When to use
Use Host Integration Pattern when your product path depends on this concern in Model Context Protocol (MCP). Prefer the simplest design that meets quality, latency, and safety budgets—and prove it with fixtures.


In [ ]:
# Demo: make "Host Integration Pattern" concrete as a checkable contract
from dataclasses import dataclass, field, asdict
import json

@dataclass
class ConceptContract:
    name: str = "Host Integration Pattern"
    notebook: str = "04-mcp-tools"
    must_have: list = field(default_factory=lambda: [
        "clear inputs/outputs",
        "failure behavior defined",
        "eval fixtures exist",
    ])
    risks: list = field(default_factory=lambda: [
        "silent quality drift",
        "unbounded cost/latency",
    ])

    def health(self) -> dict:
        return {
            "concept": self.name,
            "checks": len(self.must_have),
            "risks": len(self.risks),
            "ready_for_design_review": len(self.must_have) >= 3,
        }

contract_6 = ConceptContract()
print(json.dumps({"contract": asdict(contract_6), "health": contract_6.health()}, indent=2))


In [ ]:
# MCP-like component registry (pedagogical)
registry = {
    "resources": [{"uri": "doc://readme", "name": "README", "mimeType": "text/plain"}],
    "tools": [{"name": "search", "inputSchema": {"type": "object", "properties": {"q": {"type": "string"}}}}],
    "prompts": [{"name": "explain", "arguments": [{"name": "topic"}]}],
}

def list_caps():
    return {k: [x.get("name") or x.get("uri") for x in v] for k, v in registry.items()}

print(list_caps())


In [ ]:
# JSON-RPC style message shapes used conceptually by MCP
msg_request = {"jsonrpc": "2.0", "id": 1, "method": "tools/call", "params": {"name": "search", "arguments": {"q": "SSO"}}}
msg_response = {"jsonrpc": "2.0", "id": 1, "result": {"content": [{"type": "text", "text": "SSO allowlist..."}]}}
msg_error = {"jsonrpc": "2.0", "id": 1, "error": {"code": -32601, "message": "Method not found"}}
print(msg_request["method"], "=>", msg_response["result"]["content"][0]["text"][:40])


In [ ]:
# Demo: decision table for applying "Host Integration Pattern"
options = [
    {"option": "baseline_simple", "quality": 0.7, "cost": 1, "ops": 0.9},
    {"option": "advanced_host_integra", "quality": 0.85, "cost": 3, "ops": 0.6},
]
for o in options:
    o["utility"] = round(o["quality"] * 2 - 0.3*o["cost"] + 0.5*o["ops"], 3)
best = max(options, key=lambda x: x["utility"])
print("ranked:", sorted(options, key=lambda x: -x["utility"]))
print("prefer:", best["option"])


## Comparison Snapshot

Use this table when reviewing designs in **MCP Tools**.

| Topic | Do | Don't |
|-------|----|-------|
| What are MCP Tools? | Design carefully; measure; bound cost | Skipping eval / unbounded loops |
| Listing Tools | Design carefully; measure; bound cost | Skipping eval / unbounded loops |
| Calling Tools | Design carefully; measure; bound cost | Skipping eval / unbounded loops |
| Schemas & Validation | Design carefully; measure; bound cost | Skipping eval / unbounded loops |
| Side Effects | Design carefully; measure; bound cost | Skipping eval / unbounded loops |
| Error Model | Design carefully; measure; bound cost | Skipping eval / unbounded loops |


## Glossary / Key Terms

| Term | Meaning |
|------|---------|
| What are MCP Tools? | Key concept covered in this notebook; see its section for definition and pitfalls |
| Listing Tools | Key concept covered in this notebook; see its section for definition and pitfalls |
| Calling Tools | Key concept covered in this notebook; see its section for definition and pitfalls |
| Schemas & Validation | Key concept covered in this notebook; see its section for definition and pitfalls |
| Side Effects | Key concept covered in this notebook; see its section for definition and pitfalls |
| Error Model | Key concept covered in this notebook; see its section for definition and pitfalls |
| Host Integration Pattern | Key concept covered in this notebook; see its section for definition and pitfalls |


## Summary & Key Takeaways

- **MCP Tools** is a production concern: contracts, evals, and guardrails beat vibe-driven prompting.
- Every major topic above includes definition, motivation, mechanism, intuition, pitfalls, and usage guidance—use that checklist in design reviews.
- Prefer small, measurable demos before framework sprawl.
- Bound loops, validate tool args, and keep API keys in environment variables (`YOUR_API_KEY` is a placeholder only).
- Carry forward: connect these ideas to the next notebooks in **12-mcp**.


## Try It Yourself

1. Implement a failing test/fixture for **What are MCP Tools?**, then fix your demo until it passes.
2. Implement a failing test/fixture for **Listing Tools**, then fix your demo until it passes.
3. Implement a failing test/fixture for **Calling Tools**, then fix your demo until it passes.
4. Implement a failing test/fixture for **Schemas & Validation**, then fix your demo until it passes.
5. Implement a failing test/fixture for **Side Effects**, then fix your demo until it passes.
6. Estimate token cost for your prompt/tool trace at 1k and 100k requests/day.
7. Write a 5-row comparison of two design alternatives from this notebook; pick one with explicit criteria.
8. Red-team your solution with empty input, hostile input, and a tool/API timeout.
